# 05 — Watchlist Predictions

Predicts a rating for every unseen film on the watchlist, for the Streamlit app.

**Input:** the Letterboxd export (`watchlist.csv`), plus `data/interim/viewings.csv` and
`films_enriched.csv` for the rated history
**Output:** watchlist predictions, one row per film

The watchlist goes through the same pipeline as the rated films: matched with the same
rules (`src/tmdb.py`), given the same features, and scored by Model 4 RF refitted on all
1,192 rated viewings. Watchlist API responses are cached in **separate files**, so the
rated-film caches from `02` are never touched.

## 1. Load the watchlist

In [7]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from src.letterboxd import load_export, film_key
from src.tmdb import make_headers, match_all, AUTO_ACCEPT

load_dotenv()
export = load_export(os.getenv("LETTERBOXD_EXPORT_DIR"))

watchlist = export["watchlist"].copy()
watchlist["film_key"] = film_key(watchlist)

rated   = set(film_key(export["ratings"]))
watched = set(film_key(export["watched"]))

print(f"watchlist rows   : {len(watchlist)}")
print(f"unique film_keys : {watchlist['film_key'].nunique()}")
print(f"missing year     : {watchlist['Year'].isna().sum()}")
print(f"already rated    : {watchlist['film_key'].isin(rated).sum()}")
print(f"already watched  : {watchlist['film_key'].isin(watched).sum()}")

watchlist rows   : 4031
unique film_keys : 4022
missing year     : 8
already rated    : 0
already watched  : 0


In [3]:
dupes = watchlist[watchlist["film_key"].duplicated(keep=False)].sort_values("film_key")
print(f"{len(dupes)} rows share a film_key ({dupes['film_key'].nunique()} keys)\n")
print(dupes[["Name", "Year", "Letterboxd URI"]].to_string(index=False))

print("\nmissing year:\n")
print(watchlist[watchlist["Year"].isna()][["Name", "Letterboxd URI"]].to_string(index=False))

10 rows share a film_key (1 keys)

                    Name   Year        Letterboxd URI
              The Castle 1997.0  https://boxd.it/1sWI
              The Castle 1997.0  https://boxd.it/1Pho
                 Polaris    NaN  https://boxd.it/vMS2
       The Memory Police    NaN  https://boxd.it/scb6
         The Governesses    NaN  https://boxd.it/AdVq
           Tower Stories    NaN  https://boxd.it/nzTg
              Love Child    NaN  https://boxd.it/fAP4
The Bookie & the Bruiser    NaN  https://boxd.it/N6Oe
    Here Comes the Flood    NaN  https://boxd.it/qdjm
              Lily May B    NaN https://boxd.it/13ssm

missing year:

                    Name        Letterboxd URI
                 Polaris  https://boxd.it/vMS2
       The Memory Police  https://boxd.it/scb6
         The Governesses  https://boxd.it/AdVq
           Tower Stories  https://boxd.it/nzTg
              Love Child  https://boxd.it/fAP4
The Bookie & the Bruiser  https://boxd.it/N6Oe
    Here Comes the Flood  

In [4]:
undated  = watchlist["film_key"].isna()
collides = watchlist["film_key"].duplicated(keep=False) & ~undated

collisions = watchlist[collides].copy()

films_wl = (watchlist[~undated & ~collides]
            .rename(columns={"Name": "film_title", "Year": "film_year",
                             "Letterboxd URI": "film_uri"})
            [["film_key", "film_title", "film_year", "film_uri"]]
            .reset_index(drop=True))

print(f"excluded, no year      : {undated.sum()}")
print(f"held back, collision   : {collides.sum()} rows, {collisions['film_key'].nunique()} key(s)")
print(f"to match automatically : {len(films_wl)}")

excluded, no year      : 8
held back, collision   : 2 rows, 1 key(s)
to match automatically : 4021


In [6]:
HEADERS = make_headers(os.getenv("TMDB_TOKEN"))

WL_SEARCH_CACHE = Path("data/cache/watchlist_search_raw.json")

wl_matches = match_all(films_wl, headers=HEADERS, cache_path=WL_SEARCH_CACHE)

  100/4021 processed (100 API calls)
  200/4021 processed (200 API calls)
  300/4021 processed (300 API calls)
  400/4021 processed (400 API calls)
  500/4021 processed (500 API calls)
  600/4021 processed (600 API calls)
  700/4021 processed (700 API calls)
  800/4021 processed (800 API calls)
  900/4021 processed (900 API calls)
  1000/4021 processed (1000 API calls)
  1100/4021 processed (1100 API calls)
  1200/4021 processed (1200 API calls)
  1300/4021 processed (1300 API calls)
  1400/4021 processed (1400 API calls)
  1500/4021 processed (1500 API calls)
  1600/4021 processed (1600 API calls)
  1700/4021 processed (1700 API calls)
  1800/4021 processed (1800 API calls)
  1900/4021 processed (1900 API calls)
  2000/4021 processed (2000 API calls)
  2100/4021 processed (2100 API calls)
  2200/4021 processed (2200 API calls)
  2300/4021 processed (2300 API calls)
  2400/4021 processed (2400 API calls)
  2500/4021 processed (2500 API calls)
  2600/4021 processed (2600 API calls)
  27

In [8]:
print(f"films: {len(wl_matches)}\n")
print("confidence breakdown")
for tier, n in wl_matches["confidence"].value_counts().items():
    print(f"  {tier:12s} {n:5d}  ({n/len(wl_matches)*100:5.1f}%)")

wl_review = wl_matches[~wl_matches["confidence"].isin(AUTO_ACCEPT)]
print(f"\nauto-accepted  : {len(wl_matches) - len(wl_review)} "
      f"({(len(wl_matches) - len(wl_review)) / len(wl_matches) * 100:.1f}%)")
print(f"needs review   : {len(wl_review)}")
print(f"no match at all: {wl_matches['tmdb_id'].isna().sum()}")

films: 4021

confidence breakdown
  exact         3673  ( 91.3%)
  year_off       295  (  7.3%)
  weak            49  (  1.2%)
  close            3  (  0.1%)
  no_match         1  (  0.0%)

auto-accepted  : 3968 (98.7%)
needs review   : 53
no match at all: 1


In [9]:
print(wl_review.sort_values(["confidence", "similarity"])[
    ["film_title", "film_year", "matched_title", "matched_year",
     "confidence", "similarity", "vote_count"]
].to_string(index=False))

                                             film_title  film_year                                           matched_title  matched_year confidence  similarity  vote_count
                                                Monster     2018.0                                                 Monster        2018.0      close       1.000         1.0
                                                  Alpha     2025.0                                                   Alpha        2025.0      close       1.000       140.0
                                                Solaris     2007.0                                                 Solaris        2007.0      close       1.000         1.0
   Jeanne Dielman, 23, quai du Commerce, 1080 Bruxelles     1975.0                                                     NaN           NaN   no_match       0.000         NaN
                                  Q: The Winged Serpent     1982.0                                                       Q        1982.0    